In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
parent_dir = "/home/connorlab/Documents/GitHub/Julie/social_data/zombies_social_data/"
# Load data
aff_df = pd.read_excel(parent_dir + "zombies_feature_df_affiliation.xlsx", index_col=0)
sub_df = pd.read_excel(parent_dir + "zombies_feature_df_submission.xlsx", index_col=0)
ago_df = pd.read_excel(parent_dir + "zombies_feature_df_agonism.xlsx", index_col=0)
# 함수: "Behavior Towards 94B" → "94B"
def clean_col(colname):
    return colname.split()[-1]  # 마지막 단어만 추출

# 컬럼 이름 정리
aff_df.columns = [clean_col(c) for c in aff_df.columns]
sub_df.columns = [clean_col(c) for c in sub_df.columns]
ago_df.columns = [clean_col(c) for c in ago_df.columns]


# 공통 monkey 추출
monkeys = aff_df.index.intersection(sub_df.index).intersection(ago_df.index)
aff = np.array(aff_df)
ago = np.array(ago_df)
sub = np.array(sub_df)

In [3]:
X = aff
X

array([[ 0, 19,  9, 38, 84, 27, 13, 14,  4,  9],
       [15,  0,  9, 41,  4, 13, 19, 71, 12,  0],
       [18, 10,  0, 21,  7, 17, 49, 18,  3,  1],
       [38, 43, 24,  0, 18,  6, 31, 26, 29,  4],
       [90,  3,  8,  8,  0, 22,  8,  9,  1, 18],
       [23, 12, 17,  1, 23,  0, 23, 16,  3, 10],
       [17, 18, 43, 34, 10, 23,  0, 34,  2,  9],
       [11, 70, 18, 17,  8, 17, 33,  0,  5,  3],
       [ 3,  6,  4, 31,  2,  1,  2,  4,  0,  9],
       [ 8,  0,  0,  1, 26,  8,  2,  1,  7,  0]])

In [4]:
from analyses.spike_rate import compute_mean_spike_rate_for_windows
from analyses.enums.monkey_names import get_monkeys_by_default_order
import pandas as pd
monkey_group = "Zombies"
subject_monkey_index = 6
monkey_list = get_monkeys_by_default_order(monkey_group)
sig_windows = pd.read_pickle(
        f'/home/connorlab/Documents/GitHub/Julie/Cortana/analysis_cache/{monkey_group}_significant_windows_pANOVAorGLM_passed.pkl')
ed_sig_windows = pd.read_pickle('/home/connorlab/Documents/GitHub/Julie/Cortana/Ed and ANOVA/used_for_R01/R01_ed_list_duplicate_removed_without_full_window.pkl')
mean_spike_rate_windows = compute_mean_spike_rate_for_windows(ed_sig_windows)

Computing windowed mean spike rate: 100%|██████████| 57/57 [00:01<00:00, 46.88it/s]


In [5]:
spike_rate = mean_spike_rate_windows[mean_spike_rate_windows['MonkeyGroup']=='Zombies']
spike_rate = spike_rate[spike_rate['MonkeyName']!='NewMonkey']
# spike_rate = spike_rate[spike_rate['MonkeyName']!='7124']
spike_rate['MonkeyName'].unique()

array(['110E', '143H', '151J', '67G', '69X', '7124', '72X', '87J', '94B'],
      dtype=object)

In [6]:
X_df = pd.DataFrame(X, index=monkey_list, columns=[f'Feature{i+1}' for i in range(X.shape[1])])

In [7]:
X_df_clean = X_df.drop(index="81G")
X_df_clean

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
7124,0,19,9,38,84,27,13,14,4,9
69X,15,0,9,41,4,13,19,71,12,0
72X,18,10,0,21,7,17,49,18,3,1
94B,38,43,24,0,18,6,31,26,29,4
110E,90,3,8,8,0,22,8,9,1,18
67G,23,12,17,1,23,0,23,16,3,10
143H,11,70,18,17,8,17,33,0,5,3
87J,3,6,4,31,2,1,2,4,0,9
151J,8,0,0,1,26,8,2,1,7,0


In [8]:
from analyses.linear_regression.multiple_linear_regression import run_lasso_regression

results_df = run_lasso_regression(spike_rate, X_df_clean)
results_df

Running Lasso regressions: 100%|██████████| 57/57 [00:00<00:00, 708.30it/s]


,NeuronID,WindowStart_ms,WindowEnd_ms,R_squared,coefficients,intercept
0,AMG_2023-09-26_1_Channel.C_027_Unit 1,0.0,300.0,0.607786,"[-0.0, 0.0, 0.0, 2.324664222147351, 0.71848530...",29.019400
1,AMG_2023-09-26_2_Channel.C_011_Unit 1,700.0,1000.0,0.000000,"[-0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",1.267490
2,AMG_2023-09-26_2_Channel.C_020,50.0,150.0,0.035601,"[0.0, 0.1758495253372199, 0.0, -0.0, -0.0, 0.0...",8.432099
3,AMG_2023-09-26_3_Channel.C_007_Unit 2,100.0,200.0,0.000000,"[-0.0, -0.0, -0.0, 0.0, 0.0, -0.0, -0.0, -0.0,...",0.604938
4,AMG_2023-09-26_3_Channel.C_027_Unit 2,150.0,350.0,0.576757,"[0.7371736612228533, -0.0, -1.1149835063740567...",36.856481
5,AMG_2023-10-03_3_Channel.C_006_Unit 1,200.0,400.0,0.820206,"[-0.0, 0.0, 0.0, 0.7883865679479027, 1.9144310...",9.913580
6,AMG_2023-10-03_3_Channel.C_006_Unit 1,2150.0,2250.0,0.914466,"[-0.0, 0.8267049369410303, 0.0, 2.097501425054...",10.197531
7,AMG_2023-10-03_3_Channel.C_013_Unit 1,200.0,600.0,0.469689,"[0.0, 0.0, 0.0, -0.0, -0.0, -0.684948474591745...",6.731481
8,AMG_2023-10-03_3_Channel.C_026_Unit 1,1900.0,2000.0,0.320441,"[0.0, -0.0, 0.2950846271754267, -0.37979962253...",2.037037
9,AMG_2023-10-03_4_Channel.C_006,450.0,650.0,0.481220,"[-0.0, -0.2979728742207649, -0.0, 0.3207510006...",10.135802


In [9]:
results_df['NeuronWindowID']= (
    results_df['NeuronID'] + " (" 
    + results_df['WindowStart_ms'].astype(str) + ","
    + results_df['WindowEnd_ms'].astype(str) + ")"
)
sig_results_df = results_df[results_df['R_squared']>0.6]

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Extract the coefficients column as a new DataFrame
coeff_df = pd.DataFrame(results_df['coefficients'].tolist(),
                        index=results_df['NeuronWindowID']) 

filtered_monkey_list = [m for m in monkey_list if m not in ["81G"]]
coeff_df.columns = monkey_list

plt.figure(figsize=(12, 10))
ax = sns.heatmap(coeff_df, cmap='coolwarm', center=0, annot=True, fmt=".1f",
                 yticklabels=True, cbar=True)



# Adjust label fonts AFTER the heatmap has rendered
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

# Fix for misalignment: force re-draw with tight layout
plt.tight_layout()
plt.title("Heatmap of Regression Coefficients per Neuron", fontsize=14)
plt.xlabel("Affiliation From")
plt.ylabel("Neuron ID")
plt.show()

In [35]:
sig_coeff_df = pd.DataFrame(sig_results_df['coefficients'].tolist(),
                        index=sig_results_df['NeuronWindowID']) 
sparse_df = sig_coeff_df.replace(0, float('nan'))  # Optional: or use a threshold like < 1e-5
sparse_df.columns = monkey_list
plt.figure(figsize=(12, 10))
ax = sns.heatmap(sparse_df, cmap='coolwarm', center=0, annot=True, fmt=".1f",
                 yticklabels=True, cbar=True)

ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)
plt.tight_layout()
plt.title("Sparse Heatmap of Lasso Coefficients (Non-zero only)", fontsize=14)
plt.xlabel("Affiliation From")
plt.ylabel("Neuron ID")
plt.show()


In [31]:
selection_counts = (coeff_df != 0).sum(axis=0).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=selection_counts.values, y=selection_counts.index)
plt.title("Number of Neurons Selecting Each Feature (Non-zero Coefficients)")
plt.xlabel("Count")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# Ridge Regression

In [15]:
from analyses.linear_regression.multiple_linear_regression import run_ridge_regression

ridge_results_df = run_ridge_regression(spike_rate, X_df_clean)
ridge_results_df['NeuronWindowID']= (
    ridge_results_df['NeuronID'] + " (" 
    + ridge_results_df['WindowStart_ms'].astype(str) + ","
    + ridge_results_df['WindowEnd_ms'].astype(str) + ")"
)

Running Ridge regressions: 100%|██████████| 57/57 [00:00<00:00, 571.34it/s]


In [16]:
ridge_results_df


,NeuronID,WindowStart_ms,WindowEnd_ms,R_squared,coefficients,intercept,NeuronWindowID
0,AMG_2023-09-26_1_Channel.C_027_Unit 1,0.0,300.0,0.875254,"[-0.06781398566541427, -0.003084432468187736, ...",29.019400,"AMG_2023-09-26_1_Channel.C_027_Unit 1 (0.0,300.0)"
1,AMG_2023-09-26_2_Channel.C_011_Unit 1,700.0,1000.0,0.963928,"[-0.18601646459909532, 0.3082281323990709, 0.4...",1.267490,"AMG_2023-09-26_2_Channel.C_011_Unit 1 (700.0,1..."
2,AMG_2023-09-26_2_Channel.C_020,50.0,150.0,0.838477,"[0.005598756926974643, 1.0898215137993839, 1.2...",8.432099,"AMG_2023-09-26_2_Channel.C_020 (50.0,150.0)"
3,AMG_2023-09-26_3_Channel.C_007_Unit 2,100.0,200.0,0.945654,"[-0.32693161539069754, -0.05497374601627148, -...",0.604938,"AMG_2023-09-26_3_Channel.C_007_Unit 2 (100.0,2..."
4,AMG_2023-09-26_3_Channel.C_027_Unit 2,150.0,350.0,0.967803,"[1.745700619454517, -0.597026489478909, -2.492...",36.856481,"AMG_2023-09-26_3_Channel.C_027_Unit 2 (150.0,3..."
5,AMG_2023-10-03_3_Channel.C_006_Unit 1,200.0,400.0,0.990491,"[-0.5104971169640244, -0.1723066851729899, 0.1...",9.913580,"AMG_2023-10-03_3_Channel.C_006_Unit 1 (200.0,4..."
6,AMG_2023-10-03_3_Channel.C_006_Unit 1,2150.0,2250.0,0.995632,"[-1.403561053763676, 1.0181006752015505, 0.937...",10.197531,"AMG_2023-10-03_3_Channel.C_006_Unit 1 (2150.0,..."
7,AMG_2023-10-03_3_Channel.C_013_Unit 1,200.0,600.0,0.964785,"[-0.0731699513694563, 0.0063494147810406075, -...",6.731481,"AMG_2023-10-03_3_Channel.C_013_Unit 1 (200.0,6..."
8,AMG_2023-10-03_3_Channel.C_026_Unit 1,1900.0,2000.0,0.940384,"[0.020704795472855142, -0.9977723310688308, 1....",2.037037,"AMG_2023-10-03_3_Channel.C_026_Unit 1 (1900.0,..."
9,AMG_2023-10-03_4_Channel.C_006,450.0,650.0,0.953206,"[-0.5683744042572053, -1.2900515654143103, 0.3...",10.135802,"AMG_2023-10-03_4_Channel.C_006 (450.0,650.0)"


In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Extract the coefficients column as a new DataFrame
coeff_df = pd.DataFrame(ridge_results_df['coefficients'].tolist(),
                        index=ridge_results_df['NeuronWindowID']) 

filtered_monkey_list = [m for m in monkey_list if m not in ["81G"]]
coeff_df.columns = monkey_list

plt.figure(figsize=(12, 10))
ax = sns.heatmap(coeff_df, cmap='coolwarm', center=0, annot=True, fmt=".1f",
                 yticklabels=True, cbar=True)



# Adjust label fonts AFTER the heatmap has rendered
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

# Fix for misalignment: force re-draw with tight layout
plt.tight_layout()
plt.title("Heatmap of Regression Coefficients per Neuron", fontsize=14)
plt.xlabel("Affiliation From")
plt.ylabel("Neuron ID")
plt.show()